# Ejercicio 08 · Expediente de gobernanza y simulacro

Integra la evidencia de LAB13 en un expediente ejecutable: inventario y retención de datos,
Datasheet y Model Card trazables, RACI con traspaso entre roles, monitoreo, runbook, simulacro de
incidente y decisión final.

**Alcance.** Caso **sintético**: no representa personas, zonas ni estadísticas reales. La
gobernanza que se demuestra aquí es el **procedimiento**, no la conformidad de un tratamiento real.

**Tres capas que no se confunden**

| Capa | Qué obliga | Qué es en este expediente |
|---|---|---|
| Obligación legal | Normas vinculantes sobre protección de datos e igualdad | Ninguna verificable sobre un caso sintético: se documenta la ausencia y se prohíbe el uso con datos reales sin revisión jurídica |
| Marco voluntario | NIST AI RMF 1.0 (GOVERN, MAP, MEASURE, MANAGE) | Es el marco que estructura inventario, RACI, puertas, monitoreo y runbook |
| Recomendación ética | Juicio profesional sobre daño, proporcionalidad y apelación | Baseline sin modelo como alternativa, canal de apelación, condición de retiro y revisión humana obligatoria |

In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

from inf8239_u04.governance import load_config, validate_raci

REPORTS = ROOT / "reports"
cfg = load_config()
print("Proyecto:", ROOT)
print("Puertas declaradas:", cfg["gates"])

Proyecto: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_4/INF8239_U04_Gobernanza_DSS
Puertas declaradas: {'min_recall': 0.68, 'max_recall_gap': 0.16, 'max_fpr_gap': 0.16, 'max_latency_ms': 500, 'max_model_kb': 1024, 'min_worst_group_recall': 0.7, 'max_recall_gap_pilot': 0.1, 'min_intersectional_support': 15}


## 1. Inventario, minimización y retención

Todo campo declarado tiene origen, uso, condición de entrada al modelo, plazo de retención y
justificación; las columnas de identificadores directos se rechazan **antes** de predecir.

In [2]:
inventario = pd.DataFrame(cfg["inventory"])
print(inventario.to_string(index=False, max_colwidth=48))

figura, eje = plt.subplots(figsize=(9, 4.4))
colores = ["#0f7c7b" if u else "#94a3b8" for u in inventario["used_as_feature"]]
eje.barh(inventario["name"], inventario["retention_days"], color=colores)
for y, (d, u) in enumerate(zip(inventario["retention_days"], inventario["used_as_feature"])):
    eje.text(d + 8, y, f"{d} d" + (" · variable" if u else " · fuera del modelo"), va="center", fontsize=8)
eje.set_xlabel("Retención (días)")
eje.set_title("Inventario: retención y uso como variable · azul = entra al modelo")
eje.set_xlim(0, 1250)
eje.invert_yaxis()
eje.grid(axis="x", alpha=0.3)
figura.tight_layout()
figura.savefig(REPORTS / "lab13/inventario_retencion.png", dpi=170, bbox_inches="tight")
plt.close(figura)

print("\nMinimización:", cfg["minimization"]["excluded_from_model"], "quedan fuera del modelo")
print("Nunca permitidos:", cfg["minimization"]["prohibited_ever"])
print("Retención de artefactos:", cfg["retention"])

                name                         origin                                          purpose  used_as_feature  retention_days                                    justification
             case_id territorial_risk_synthetic.csv          trazabilidad interna del caso sintetico            False             730 identificador del registro sintetico; no cont...
       received_date territorial_risk_synthetic.csv             corte temporal y deteccion de deriva            False             730 fecha sintetica de recepcion; no se usa como ...
                zone territorial_risk_synthetic.csv                              variable predictora             True             365 contexto territorial sintetico, disponible al...
            age_band territorial_risk_synthetic.csv   variable predictora y auditoria interseccional             True             365  banda de edad sintetica, disponible al predecir
  previous_incidents territorial_risk_synthetic.csv                              vari

## 2. RACI y traspaso entre roles

Cada actividad tiene **un solo responsable final (A)** y al menos un responsable de ejecución (R).
El traspaso entre roles es explícito: quien entrega un artefacto lo entrega a un rol que puede
rechazarlo.

In [3]:
raci = pd.read_csv(ROOT / "student_inputs/raci.csv")
validate_raci(raci)
print(f"Actividades con RACI válido: {len(raci)}")
print(raci.to_string(index=False, max_colwidth=34))

conteo = raci["A"].value_counts()
figura, eje = plt.subplots(figsize=(9, 4))
eje.barh(conteo.index, conteo.values, color="#b45309")
for y, v in enumerate(conteo.values):
    eje.text(v + 0.06, y, str(v), va="center", fontsize=9)
eje.set_xlabel("Actividades con ese rol como responsable final (A)")
eje.set_title("Distribución de responsabilidad final por rol")
eje.set_xlim(0, conteo.max() + 0.8)
eje.grid(axis="x", alpha=0.3)
figura.tight_layout()
figura.savefig(REPORTS / "lab13/raci_responsables.png", dpi=170, bbox_inches="tight")
plt.close(figura)

Actividades con RACI válido: 12
                          activity                      R                                 A                                 C                        I
    Data quality and data contract          Data Engineer                     Product Owner                    Data Scientist Affected representatives
Privacy review and prohibited c...          Data Engineer Legal and Data Protection Officer                    Data Scientist Affected representatives
Model validation and reproducib...         Data Scientist                     Product Owner                     Data Engineer                    Users
Fairness audit (global, group a...         Data Scientist            Responsible for Equity Legal and Data Protection Officer Affected representatives
Impact assessment and alternati...         Data Scientist               Institutional Owner            Responsible for Equity Affected representatives
Deployment approval of the gove...            ML Engineer     

## 3. Puertas de calidad y decisión gobernada

Las cinco primeras puertas son las del proyecto base; las tres siguientes traducen la tolerancia
acordada en el LAB12. **Una puerta fallida produce REVIEW o STOP y nunca se oculta.**

In [4]:
decision = json.loads((REPORTS / "lab13/deployment_decision.json").read_text(encoding="utf-8"))
print("Decisión:", decision["decision"]["status"])
print("Puertas fallidas:", decision["decision"]["failed"])
print("Métricas:", decision["metrics"])
print("Brechas:", decision["gaps"])
print("Recursos:", decision["resources"])

figura, eje = plt.subplots(figsize=(9, 4.4))
puertas = decision["decision"]["checks"]
etiquetas = list(puertas)
valores = [1 if puertas[k] else 0 for k in etiquetas]
eje.barh(etiquetas, valores, color=["#15803d" if v else "#b91c1c" for v in valores])
for y, (k, v) in enumerate(puertas.items()):
    eje.text(1.02, y, "correcta" if v else "FALLA", va="center", fontsize=9,
             color="#15803d" if v else "#b91c1c")
eje.set_xlim(0, 1.35)
eje.set_xticks([])
eje.set_title("Puertas de calidad del despliegue · verde = correcta, rojo = falla")
eje.invert_yaxis()
figura.tight_layout()
figura.savefig(REPORTS / "lab13/puertas_calidad.png", dpi=170, bbox_inches="tight")
plt.close(figura)

Decisión: REVIEW
Puertas fallidas: ['worst_group_recall', 'recall_gap_pilot', 'intersectional_support']
Métricas: {'accuracy': 0.71875, 'recall': 0.717391304347826, 'worst_group_recall': 0.6530612244897959, 'worst_group': 'G1'}
Brechas: {'recall_gap': 0.13763644992880875, 'fpr_gap': 0.125}
Recursos: {'training_seconds': 0.019056299999647308, 'model_kb': 3.5634765625, 'latency_ms': 0.01604986875349823, 'min_intersectional_support': 8}


## 4. Simulacro de incidente

Escenario: la puerta de equidad falla (brecha de recall 0,1376 > 0,10 de tolerancia de piloto). El
sistema degrada a REVIEW, se congela la priorización automática y se vuelve al baseline sin modelo.

In [5]:
postmortem = (REPORTS / "lab13/INCIDENT_POSTMORTEM.md").read_text(encoding="utf-8")
for linea in postmortem.splitlines():
    if linea.startswith("- "):
        clave = linea[2:].split(":")[0]
        print(f"· {clave}: {linea[2:].split(':', 1)[1][:110].strip()}...")

· detected_at: 2026-11-02T12:42:01.013756+00:00...
· signal: recall gap exceeded...
· signal_detail: la brecha de recall entre grupos (0.1376) supera la tolerancia de piloto acordada en el LAB12 (0.10); ademas...
· detection: comparacion mensual de las metricas contra los umbrales de configs/governance.yml (monitoring.alert_threshold...
· severity: media (el sistema no decide de forma autonoma y no se han producido acciones sobre personas; el riesgo es de...
· containment: la decision del pipeline pasa a REVIEW y se congela el despliegue; la cola deja de ordenar automaticamente y...
· authority: Institutional Owner (rol accountable unico) ordena la contencion; ML Engineer ejecuta el rollback; Legal and...
· evidence: se preservan logs de ejecucion, version del modelo (models/logistic.joblib), metricas por grupo e interseccio...
· communication: el Institutional Owner informa al equipo de operacion y a los representantes de las personas afectadas a trav...
· rollback: retorno a la versi

## 5. Monitoreo y calidad documental

| Evidencia | Archivo | Verificación |
|---|---|---|
| Decisión gobernada | `reports/lab13/deployment_decision.json` | `scripts/lab13_governance.py` |
| Métricas por grupo del DSS | `reports/lab13/fairness_by_group.csv` | Fairlearn `MetricFrame` |
| Inventario y retención | `configs/governance.yml`, `docs/INVENTARIO_Y_RETENCION.md` | `tests/test_governance.py` |
| RACI | `student_inputs/raci.csv` | `validate_raci` |
| Model Card y Datasheet | `docs/MODEL_CARD.md`, `docs/DATASET_CARD.md` | cifras contra `reports/` |
| Runbook | `docs/RUNBOOK.md` | simulacro `scripts/simulate_incident.py` |
| Decisión de impacto (LAB11) | `reports/lab11/DECISION_RECORD.md` | `scripts/lab11_impact.py` |
| Decisión de equidad (LAB12) | `student_inputs/lab12_decision.yml` | `scripts/validate_lab12_decision.py` |

In [6]:
resumen = {
    "decision_final": "LIMITAR",
    "decision_maquina": decision["decision"]["status"],
    "puertas_fallidas": decision["decision"]["failed"],
    "gates_base": {k: cfg["gates"][k] for k in ["min_recall", "max_recall_gap", "max_fpr_gap",
                                                "max_latency_ms", "max_model_kb"]},
    "gates_lab12": {k: cfg["gates"][k] for k in ["min_worst_group_recall", "max_recall_gap_pilot",
                                                 "min_intersectional_support"]},
    "baseline_sin_modelo": "prioridad por severidad reportada con revision humana",
    "supervision_humana": "obligatoria; el DSS no decide",
    "proxima_revision": "2027-01-15",
}
(REPORTS / "lab13/ejercicio08_resumen.json").write_text(json.dumps(resumen, indent=2), encoding="utf-8")
print(json.dumps(resumen, indent=2))

{
  "decision_final": "LIMITAR",
  "decision_maquina": "REVIEW",
  "puertas_fallidas": [
    "worst_group_recall",
    "recall_gap_pilot",
    "intersectional_support"
  ],
  "gates_base": {
    "min_recall": 0.68,
    "max_recall_gap": 0.16,
    "max_fpr_gap": 0.16,
    "max_latency_ms": 500,
    "max_model_kb": 1024
  },
  "gates_lab12": {
    "min_worst_group_recall": 0.7,
    "max_recall_gap_pilot": 0.1,
    "min_intersectional_support": 15
  },
  "baseline_sin_modelo": "prioridad por severidad reportada con revision humana",
  "supervision_humana": "obligatoria; el DSS no decide",
  "proxima_revision": "2027-01-15"
}


## 6. Interpretación y decisión final

**Observación.** Con los umbrales por defecto del proyecto base (brecha de recall ≤ 0,16) el pipeline
daría **GO**: recall 0,7174, brecha 0,1376, FPR 0,1250, latencia 0,016 ms y modelo de 3,56 KB
cumplen las cinco puertas del base. Pero la tolerancia que el estudiante acordó en el LAB12 —brecha de
recall ≤ 0,10 y recall del grupo con peor desempeño ≥ 0,70— **no** se cumple: el recall de G1 es
0,6531 y hay dos celdas interseccionales con soporte insuficiente (n=8 y n=11). Con esa tolerancia,
tres puertas fallan y la decisión del sistema es **REVIEW**.

**Evidencia.** `reports/lab13/deployment_decision.json` (checks y puertas fallidas),
`reports/lab12/logistic/fairness_by_group.csv` y `fairness_intersectional.csv` (conteos),
`reports/lab13/INCIDENT_POSTMORTEM.md` (simulacro) y `student_inputs/raci.csv` (responsabilidades).

**Interpretación.** La brecha **no prueba discriminación causal**: el atributo sensible no entra al
modelo y parte de la diferencia es la tasa base de positivos (G2 63,9 % frente a G1 49,6 %). Pero
tampoco puede descartarse con una sola partición, una sola semilla, sin intervalos de confianza y
con soporte interseccional insuficiente: por eso la decisión no se apoya en la métrica, sino en la
tolerancia acordada y en el riesgo residual alto (R07, presión para decidir sin revisión humana).

**Decisión final: LIMITAR.** No desplegar de forma autónoma. Mantener el DSS en modo REVIEW con
revisión humana obligatoria, el baseline sin modelo como camino operativo seguro, el simulacro de
incidente y el canal de apelación activos, y el sistema congelado hasta que (1) se repita la
auditoría con un periodo posterior y con intervalos de confianza, (2) se cuantifique el coste real
del falso negativo para fijar la tolerancia con evidencia, y (3) se amplíe el soporte interseccional o
se retiren esas métricas del tablero. Condiciones de retiro y propiedad por rol en el runbook; la
**obligación legal** no se sustituye con el marco voluntario ni con la recomendación ética: cualquier
uso con datos reales exige revisión jurídica previa.